# A4 — Transfer Learning & Fine-Tuning (CPU-Friendly)
## Data Set: Oxford-IIIT Pet 
- #### Feature Extraction vs Fine-Tuning | Performance Comparison
### Models: VGG16 • ResNet50 • DenseNet121
#### Solution Notebook

**Hardware assumption:** CPU-only laptops/PC are acceptable (training time may vary).  
**Dataset:** Oxford-IIIT Pet (37 classes)  
**Recommended settings:** `IMG_SIZE=(128,128)`, `BATCH_SIZE=32`, `EPOCHS=5-6` (CPU-friendly)

---

This solution notebook provides a clean reference implementation for:
- Building a reproducible `tf.data` pipeline
- Training **frozen-backbone** models (feature extraction)
- Running **one controlled fine-tuning** experiment
- Comparing results (accuracy, training time, parameter counts)


## Q0 — Setup (Ungraded)
#### Import libraries, set seeds, and verify TensorFlow / TFDS.

In [ ]:
import os, time, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow.keras import layers
from tensorflow.keras.applications import resnet, vgg16, densenet

print("TensorFlow:", tf.__version__)
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

## ✅ Student Instructions (Start Here)

Your work begins in the **next code cells (Q1–Q9)** and continues by answering questions in the **Markdown cells (Q10–Q13)**. These correspond to the questions listed in the assignment description on Canvas. Complete each cell by following the instructions provided in the **preceding Markdown cells**.

Tasks:
- This assignment focuses on **transfer learning** with pretrained CNN backbones.
- You will train **three frozen-backbone models** for a fair comparison:
  - **VGG16** (frozen base)
  - **ResNet50** (frozen base)
  - **DenseNet121** (frozen base)
- Then run **one controlled fine-tuning experiment** (unfreeze a small portion of one backbone with a smaller learning rate).
- Keep your training CPU-feasible (use the recommended settings unless you have a GPU).

## Q1 — Load Dataset & Inspect
Use TFDS: `oxford_iiit_pet` and inspect shapes/classes.
### Student Tasks

- Load the Oxford-IIIT Pet dataset and split into ds_train (80%), ds_val (20%), and ds_test.

- Extract number of classes and class names from ds_info.

- Display one sample: image shape, label index, and class name.

In [ ]:
#============================================================
#Question Q1 — Load Oxford-IIIT Pet Dataset (Fill in the Blanks)
#============================================================
#Complete the TODO sections to:
#1) Load the Oxford-IIIT Pet dataset using TensorFlow Datasets
#2) Create train/validation/test splits
#3) Extract class names and number of classes
#4) Print dataset information
#5) Inspect one example image and label
#============================================================

# TODO 1: Load dataset with train/validation/test splits
(ds_train, ds_val, ds_test), ds_info = tfds.__________(
    "oxford_iiit_pet",
    split=["____________", "____________", "____________"],
    as_supervised=__________,
    with_info=__________,
)

# TODO 2: Get number of classes
num_classes = ds_info.features["__________"].__________

# TODO 3: Get class names
class_names = ds_info.features["__________"].__________

# TODO 4: Print dataset information
print("Num classes:", __________)
print("Example classes:", __________[:____])

# TODO 5: Inspect one example from the training set
for x, y in ds_train.__________(____):
    print("Raw image shape:", x.__________, 
          "| label:", int(__________), 
          "| class:", class_names[int(__________)])

## Q2 — Preprocessing & `tf.data` Pipeline
Resize to **128×128**, normalize, add light augmentation for training.

### Student Tasks

- Set image size and batch size.

- Preprocess images by resizing and normalizing.

- Apply data augmentation to the training dataset.

- Create batched and prefetched train, validation, and test pipelines.

In [ ]:
#============================================================
#Question Q2 — Image Preprocessing & Data Pipeline (Fill in the Blanks)
#============================================================
#Complete the TODO sections to:
#1) Define image size (128×128) and batch size (32 to 64)
#2) Implement preprocessing (resize + normalization)
#3) Implement simple data augmentation
#4) Build TensorFlow data pipelines
#5) Shuffle, batch, and prefetch the datasets
#============================================================

# TODO 1: Define image size and batch size
IMG_SIZE = (____, ____)
BATCH_SIZE = ____
AUTOTUNE = tf.data.__________

# ------------------------------------------------------------
# TODO 2: Preprocessing function
# Resize images and normalize pixel values to [0,1]
# ------------------------------------------------------------
@tf.function
def preprocess(image, label):
    
    # Resize image
    image = tf.image.__________(image, IMG_SIZE)
    
    # Convert to float32
    image = tf.cast(image, tf.__________)
    
    # Normalize pixels
    image = image / ______
    
    return image, label

# ------------------------------------------------------------
# TODO 3: Data augmentation function
# ------------------------------------------------------------
@tf.function
def augment(image, label):
    
    # Random horizontal flip
    image = tf.image.____________________(image)
    
    # Random brightness
    image = tf.image.____________________(image, max_delta=____)
    
    return image, label

# ------------------------------------------------------------
# TODO 4: Apply preprocessing and augmentation
# ------------------------------------------------------------
train_ds = ds_train.__________(preprocess, num_parallel_calls=AUTOTUNE)\
                   .__________(augment, num_parallel_calls=AUTOTUNE)

val_ds   = ds_val.__________(preprocess, num_parallel_calls=AUTOTUNE)

test_ds  = ds_test.__________(preprocess, num_parallel_calls=AUTOTUNE)

# ------------------------------------------------------------
# TODO 5: Shuffle, batch, and prefetch
# ------------------------------------------------------------
train_ds = train_ds.__________(____, seed=SEED)\
                   .__________(BATCH_SIZE)\
                   .__________(AUTOTUNE)

val_ds   = val_ds.__________(BATCH_SIZE)\
                 .__________(AUTOTUNE)

test_ds  = test_ds.__________(BATCH_SIZE)\
                  .__________(AUTOTUNE)

print("Ready:", train_ds, val_ds, test_ds)

## Q3 — Utilities: Compile / Train / Evaluate / Count Params

### Student Tasks

- Compile the model using the Adam optimizer, sparse categorical cross-entropy loss, and accuracy metric.

- Train the model for several epochs and measure the training time.

- Evaluate the trained model on the validation and test datasets and report accuracy.

In [ ]:
# ============================================================
# Question Q3 — Model Training Utilities (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Compile a model using the Adam optimizer
# 2) Define the loss function for multi-class classification
# 3) Track accuracy during training
# 4) Compute total model parameters
# 5) Train the model and measure training time
# 6) Evaluate the model on validation and test datasets
# ============================================================

import numpy as np
import time
import tensorflow as tf

# ------------------------------------------------------------
# TODO 1: Compile the model
# ------------------------------------------------------------
def compile_model(model, lr=1e-3):

    model.compile(

        optimizer=tf.keras.optimizers.__________(learning_rate=____),

        loss=tf.keras.losses.________________________________(from_logits=_____),

        metrics=["__________"],
    )

    return model


# ------------------------------------------------------------
# TODO 2: Compute total parameters of the model
# ------------------------------------------------------------
def total_params(model):

    return int(np.sum([np.prod(v.__________) for v in model.________________])) + \
           int(np.sum([np.prod(v.__________) for v in model.________________]))


# ------------------------------------------------------------
# TODO 3: Train the model and measure training time
# ------------------------------------------------------------
def train_and_time(model, run_name, epochs=8):

    t0 = time.__________()

    hist = model.__________(
        train_ds,
        validation_data=__________,
        epochs=__________,
        verbose=____
    )

    dt = time.__________() - t0

    return hist, dt


# ------------------------------------------------------------
# TODO 4: Evaluate the model
# ------------------------------------------------------------
def evaluate(model, name):

    v = model.__________(__________, verbose=____)

    t = model.__________(__________, verbose=____)

    print(f"{name} | Val acc: {v[1]:.4f} | Test acc: {t[1]:.4f}")

    return {
        "val_loss": v[0],
        "val_acc": v[1],
        "test_loss": t[0],
        "test_acc": t[1],
    }

## Q4 — Backbones & Transfer Model Builder

We will compare three pretrained ImageNet backbones using the **same head design** (GAP + Dropout + Dense):
- **VGG16**
- **ResNet50**
- **DenseNet121**

For feature extraction, keep the backbone **frozen** (`train_base=False`). For fine-tuning, unfreeze a small number of top layers with a smaller learning rate.

### Student Tasks

- Implement a function to load a pretrained backbone model (VGG16, ResNet50, or DenseNet121).

- Build a transfer learning model by adding a Global Average Pooling and classification layer.

- Optionally unfreeze the last layers of the backbone to perform fine-tuning with a smaller learning rate.


In [ ]:
# ============================================================
# Question Q4 — Transfer Learning Model Construction (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Build a backbone model using pretrained ImageNet weights
# 2) Support VGG16, ResNet50, and DenseNet121 architectures
# 3) Attach a classifier head using Global Average Pooling
# 4) Create a full transfer learning model
# 5) Implement fine-tuning by unfreezing part of the backbone
# ============================================================

# ------------------------------------------------------------
# TODO 1: Build backbone model
# ------------------------------------------------------------
def build_backbone(backbone_name: str):
    """Return (base_model, preprocess_fn) for a supported backbone."""

    name = backbone_name.__________()

    if name == "vgg16":
        base = tf.keras.applications.__________(
            weights="__________",
            include_top=__________,
            input_shape=IMG_SIZE + (___,)
        )

        preprocess_fn = vgg16.____________________

    elif name == "resnet50":
        base = tf.keras.applications.__________(
            weights="__________",
            include_top=__________,
            input_shape=IMG_SIZE + (___,)
        )

        preprocess_fn = resnet.____________________

    elif name in ["densenet121", "dense121"]:
        base = tf.keras.applications.__________(
            weights="__________",
            include_top=__________,
            input_shape=IMG_SIZE + (___,)
        )

        preprocess_fn = densenet.____________________

    else:
        raise ValueError(f"Unknown backbone: {backbone_name}")

    return base, preprocess_fn


# ------------------------------------------------------------
# TODO 2: Build transfer learning model
# ------------------------------------------------------------
def build_transfer_model(backbone_name="resnet50", train_base=False, dropout=0.2):
    """Backbone + GAP head. Returns (model, base)."""

    base, preprocess_fn = build_backbone(backbone_name)

    # Freeze or unfreeze backbone
    base.__________ = bool(__________)

    inputs = tf.keras.Input(shape=IMG_SIZE + (___,))

    # Apply preprocessing function
    x = preprocess_fn(inputs * ______)

    # Forward pass through backbone
    x = base(x, training=__________)

    # Global Average Pooling
    x = layers.____________________()(x)

    # Dropout regularization
    x = layers.____________________(dropout)(x)

    # Output classification layer
    outputs = layers.__________(num_classes, activation="__________")(x)

    # Build final model
    model = tf.keras.__________(
        inputs,
        outputs,
        name=f"{backbone_name}_gap_{'ft' if train_base else 'fz'}"
    )

    return model, base


# ------------------------------------------------------------
# TODO 3: Fine-tune the backbone
# ------------------------------------------------------------
def fine_tune(model, base, n_unfreeze=30, lr=1e-5):
    """Unfreeze last layers of backbone and recompile model."""

    # Enable backbone training
    base.__________ = ______

    if n_unfreeze is not None:
        for layer in base.layers[:-int(__________)]:
            layer.__________ = ______

    # Recompile model with smaller learning rate
    model = compile_model(model, lr=____)

    return model

## Q5 — Model A: **VGG16** Transfer Learning (Frozen Base)
Train only a small classification head first (feature extraction).

### Student Tasks

- Build a transfer learning model using a frozen VGG16 backbone with a Global Average Pooling head.

- Compile the model and display the model summary.

- Train the model and evaluate its validation and test accuracy.

In [ ]:
# ============================================================
# Question Q5 — Train VGG16 Transfer Learning Model (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Build a transfer learning model using the VGG16 backbone
# 2) Freeze the backbone during initial training
# 3) Compile the model with the provided compile_model() function
# 4) Train the model and record training time
# 5) Evaluate the trained model on validation and test datasets
# ============================================================

# ------------------------------------------------------------
# TODO 1: Build the VGG16 transfer learning model
# ------------------------------------------------------------
vgg_model, vgg_base = build_transfer_model(
    backbone_name="__________",
    train_base=__________,
    dropout=____
)

# ------------------------------------------------------------
# TODO 2: Compile the model
# ------------------------------------------------------------
vgg_model = compile_model(
    vgg_model,
    lr=____
)

# ------------------------------------------------------------
# TODO 3: Display model summary
# ------------------------------------------------------------
vgg_model.__________()


# ------------------------------------------------------------
# TODO 4: Train the model
# ------------------------------------------------------------
history_vgg_fz, time_vgg_fz = train_and_time(
    vgg_model,
    "__________",
    epochs=____
)

# ------------------------------------------------------------
# TODO 5: Evaluate the trained model
# ------------------------------------------------------------
res_vgg_fz = evaluate(
    vgg_model,
    "__________"
)

## Q6 — Model B: **ResNet50** Transfer Learning (Frozen Base)
Same head design as VGG16 model for a fair backbone comparison.

### Student Tasks

- Build a transfer learning model using a frozen ResNet50 backbone with a Global Average Pooling head.

- Compile the model and display the model summary.

- Train the model and evaluate its validation and test accuracy.

In [ ]:
# ============================================================
# Question Q6 — Train ResNet50 Transfer Learning Model (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Build a transfer learning model using the ResNet50 backbone
# 2) Freeze the backbone during initial training
# 3) Compile the model using compile_model()
# 4) Train the model and record training time
# 5) Evaluate the trained model on validation and test datasets
# ============================================================

# ------------------------------------------------------------
# TODO 1: Build the ResNet50 transfer learning model
# ------------------------------------------------------------
resnet_model, resnet_base = build_transfer_model(
    backbone_name="__________",
    train_base=__________,
    dropout=____
)

# ------------------------------------------------------------
# TODO 2: Compile the model
# ------------------------------------------------------------
resnet_model = compile_model(
    resnet_model,
    lr=____
)

# ------------------------------------------------------------
# TODO 3: Display model summary
# ------------------------------------------------------------
resnet_model.__________()


# ------------------------------------------------------------
# TODO 4: Train the model
# ------------------------------------------------------------
history_resnet_fz, time_resnet_fz = train_and_time(
    resnet_model,
    "__________",
    epochs=____
)

# ------------------------------------------------------------
# TODO 5: Evaluate the trained model
# ------------------------------------------------------------
res_resnet_fz = evaluate(
    resnet_model,
    "__________"
)

## Q7 — Model C: **DenseNet121** Transfer Learning (Frozen Base)
Train a DenseNet121 backbone with the same GAP head design for a fair comparison against VGG16 and ResNet50.

### Student Tasks

- Build a transfer learning model using a frozen DenseNet121 backbone.

- Compile the model and display the model summary.

- Train the model and evaluate its validation and test accuracy.


In [ ]:
# ============================================================
# Question Q7 — Train DenseNet121 Transfer Learning Model (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Build a transfer learning model using the DenseNet121 backbone
# 2) Freeze the backbone during initial training
# 3) Compile the model using compile_model()
# 4) Train the model and measure training time
# 5) Evaluate the trained model on validation and test datasets
# ============================================================

# ------------------------------------------------------------
# TODO 1: Build the DenseNet121 transfer learning model
# ------------------------------------------------------------
densenet_model, densenet_base = build_transfer_model(
    backbone_name="__________",    # e.g., densenet121
    train_base=__________,
    dropout=____
)

# ------------------------------------------------------------
# TODO 2: Compile the model
# ------------------------------------------------------------
densenet_model = compile_model(
    densenet_model,
    lr=____
)

# ------------------------------------------------------------
# TODO 3: Display model summary
# ------------------------------------------------------------
densenet_model.__________()


# ------------------------------------------------------------
# TODO 4: Train the model
# ------------------------------------------------------------
history_densenet_fz, time_densenet_fz = train_and_time(
    densenet_model,
    "__________",
    epochs=____
)

# ------------------------------------------------------------
# TODO 5: Evaluate the trained model
# ------------------------------------------------------------
res_densenet_fz = evaluate(
    densenet_model,
    "__________"
)

## Q8 — Fine-Tuning Experiment (One Controlled Change)

Fine-tune **all three backbone models (VGG16, ResNet50, and DenseNet121)** by unfreezing only the **top N layers**.

Report whether performance **improves, stays similar, or degrades**, and briefly explain why.

**Recommended**
- Start by unfreezing the **last 10–30 layers**
- Use a **smaller learning rate** (e.g., `1e-5`)
- Use **2-3 epochs**

### Student Tasks

- Fine-tune the **VGG16, ResNet50, and DenseNet121** models by unfreezing the top layers of each backbone.

- Train each fine-tuned model for several epochs.

- Evaluate the validation and test accuracy of each fine-tuned model.

## Q8(a) — Fine-tune VGG16

In [ ]:
# ============================================================
# Question Q8(a) — Fine-Tune VGG16 Transfer Learning Model (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Fine-tune the VGG16 backbone by unfreezing the last layers
# 2) Re-compile the model with a smaller learning rate
# 3) Train the fine-tuned model
# 4) Evaluate the model after fine-tuning
# ============================================================

# ------------------------------------------------------------
# TODO 1: Fine-tune the VGG16 backbone
# ------------------------------------------------------------
vgg_model = fine_tune(
    vgg_model,
    vgg_base,
    n_unfreeze=____,
    lr=____
)

# ------------------------------------------------------------
# TODO 2: Train the fine-tuned model
# ------------------------------------------------------------
history_vgg_ft, time_vgg_ft = train_and_time(
    vgg_model,
    "__________",
    epochs=____
)

# ------------------------------------------------------------
# TODO 3: Evaluate the fine-tuned model
# ------------------------------------------------------------
res_vgg_ft = evaluate(
    vgg_model,
    "__________"
)

## Q8(b) — Fine-tune ResNet50

In [ ]:
# ============================================================
# Question Q8(b) — Fine-Tune ResNet50 Transfer Learning Model (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Fine-tune the ResNet50 backbone by unfreezing the last layers
# 2) Re-compile the model with a smaller learning rate
# 3) Train the fine-tuned model
# 4) Evaluate the model after fine-tuning
# ============================================================

# ------------------------------------------------------------
# TODO 1: Fine-tune the ResNet50 backbone
# ------------------------------------------------------------
resnet_model = fine_tune(
    resnet_model,
    resnet_base,
    n_unfreeze=____,
    lr=____
)

# ------------------------------------------------------------
# TODO 2: Train the fine-tuned model
# ------------------------------------------------------------
history_resnet_ft, time_resnet_ft = train_and_time(
    resnet_model,
    "__________",
    epochs=____
)

# ------------------------------------------------------------
# TODO 3: Evaluate the fine-tuned model
# ------------------------------------------------------------
res_resnet_ft = evaluate(
    resnet_model,
    "__________"
)

## Q8(c) — Fine-tune DenseNet121

In [ ]:
# ============================================================
# Question Q8(c) — Fine-Tune DenseNet121 Transfer Learning Model (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Fine-tune the DenseNet121 backbone by unfreezing the last layers
# 2) Re-compile the model with a smaller learning rate
# 3) Train the fine-tuned model
# 4) Evaluate the model after fine-tuning
# ============================================================

# ------------------------------------------------------------
# TODO 1: Fine-tune the DenseNet121 backbone
# ------------------------------------------------------------
densenet_model = fine_tune(
    densenet_model,
    densenet_base,
    n_unfreeze=____,
    lr=____
)

# ------------------------------------------------------------
# TODO 2: Train the fine-tuned model
# ------------------------------------------------------------
history_densenet_ft, time_densenet_ft = train_and_time(
    densenet_model,
    "__________",
    epochs=____
)

# ------------------------------------------------------------
# TODO 3: Evaluate the fine-tuned model
# ------------------------------------------------------------
res_densenet_ft = evaluate(
    densenet_model,
    "__________"
)

## Q9 — Performance Comparison Table

#### Create a compact table comparing **frozen-backbone** models:

- VGG16 (frozen)
- ResNet50 (frozen)
- DenseNet121 (frozen)

#### Also include **fine-tuned** results for all three backbone.

### Student Tasks

- Create a comparison table summarizing the results of all models.

- Include validation accuracy, test accuracy, and training time.

- Compare frozen and fine-tuned models and identify the best-performing model.


In [ ]:
# ============================================================
# Question Q9 — Compare Model Results (Fill in the Blanks)
# ============================================================
# Complete the TODO sections to:
# 1) Store evaluation results for frozen models
# 2) Append results for fine-tuned models if they exist
# 3) Create a comparison table using pandas
# 4) Sort models by test accuracy
# ============================================================

# ------------------------------------------------------------
# TODO 1: Create result rows for frozen models
# ------------------------------------------------------------
rows = [

    {"Model": "__________",       
     "Val acc": res_vgg_fz["__________"],       
     "Test acc": res_vgg_fz["__________"],       
     "Train time (s)": __________},

    {"Model": "__________",    
     "Val acc": res_resnet_fz["__________"],    
     "Test acc": res_resnet_fz["__________"],    
     "Train time (s)": __________},

    {"Model": "__________", 
     "Val acc": res_densenet_fz["__________"],  
     "Test acc": res_densenet_fz["__________"],  
     "Train time (s)": __________},
]

# ------------------------------------------------------------
# TODO 2: Append results for fine-tuned models if available
# ------------------------------------------------------------

if "__________" in globals():
    rows.append({
        "Model": "__________",
        "Val acc": res_vgg_ft["__________"],
        "Test acc": res_vgg_ft["__________"],
        "Train time (s)": __________
    })

if "__________" in globals():
    rows.append({
        "Model": "__________",
        "Val acc": res_resnet_ft["__________"],
        "Test acc": res_resnet_ft["__________"],
        "Train time (s)": __________
    })

if "__________" in globals():
    rows.append({
        "Model": "__________",
        "Val acc": res_densenet_ft["__________"],
        "Test acc": res_densenet_ft["__________"],
        "Train time (s)": __________
    })


# ------------------------------------------------------------
# TODO 3: Create comparison dataframe
# ------------------------------------------------------------
df = pd.__________(rows)


# ------------------------------------------------------------
# TODO 4: Sort models by test accuracy
# ------------------------------------------------------------
df = df.__________________("__________", ascending=__________)

df

## Short Written Questions

## **Q10** — What is the difference between *feature extraction* and *fine-tuning* in transfer learning?  
**Answer:** Type your answer here....

## **Q11** — Why should the learning rate typically be lower during fine-tuning than during feature extraction?  

**Answer:** Type your answer here....

## **Q12** — Give one reason why these backbones may behave differently on Oxford-IIIT Pet.  

**Answer:** Type your answer here....

## **Q13** — Under what conditions can fine-tuning reduce performance compared to a frozen backbone?  

**Answer:** Type your answer here....

### 🎉 Congratulations!

You have successfully completed **A4-TL**. Excellent work exploring and comparing **RL**, specifically **VGG16**, **VGG16**, and **DenseNet121**. Analyzing how **TL, and fine-tuning** affect performance on the **Oxford-IIIT Pet** under **CPU-only training constraints**.

### **Submission Instructions**

Please submit a **GitHub repository link** on Canvas that contains:
- The **completed Jupyter notebook**
- Notebook runs **top-to-bottom** without errors

Before submitting, ensure that:
- All **code cells (Q1–Q9)** have been executed successfully
- All **Markdown responses (Q10–Q13)** have been completed
- The notebook is **saved after execution** so that outputs are visible

Once verified, **push the final version to GitHub** and submit the repository link on Canvas.